# ChronoQuant Controller

Főbb scripteket indító vezérlő notebook. Minden szekció egy önálló feladatot hajt végre — csak a szükséges cell-t futtasd.

---

## Doc Merge + Rendering

Összegyűjti a `_doc_/` összes `.md` és `.ipynb` fájlját egyetlen Quarto-renderable notebookban, majd HTML-t renderel belőle.

**Output:** `_chronoquant_docs.ipynb` + `_chronoquant_docs.html`

In [ ]:
import subprocess
import sys

OUT = "_chronoquant_docs.ipynb"
# OUT = "_chronoquant_docs_draft.ipynb"  # alternatív kimeneti fájlnév

# 1. Merge: _doc_/*.md + _doc_/*.ipynb → egyetlen notebook
subprocess.run(
    [sys.executable, "analyst/doc_renderer/build_doc_notebook.py", "--out", OUT],
    check=True,
)

# 2. Render: notebook → HTML
subprocess.run(
    ["quarto", "render", OUT, "--to", "html"],
    check=True,
)

print(f"Done: {OUT.replace('.ipynb', '.html')}")

---

## Model Training Pipeline

Egy modell teljes fejlesztési pipeline-ját futtatja sorban: `setup → feature_engineering → search → train`.

**Lépések:**
- `setup` — artifact mappa + `manifest.json` létrehozása
- `feature_engineering` — feature quality notebook futtatása papermill-lel, HTML render
- `search` — LightGBM hyperparameter keresés Optunával (`smoke` / `explore` / `refine`)
- `train` — végső modell fittelése, `model.pkl`, `features.json`, `params.json`, `sample_oos.parquet`

**Elérhető modellek:** `l` = long, `s` = short; évek: 2021–2025

In [ ]:
import subprocess

# --- Modell kiválasztása ---
MODEL = "lgbm_solusdt_l_fw60_2021"
# MODEL = "lgbm_solusdt_l_fw60_2022"
# MODEL = "lgbm_solusdt_l_fw60_2023"
# MODEL = "lgbm_solusdt_l_fw60_2024"
# MODEL = "lgbm_solusdt_l_fw60_2025"
# MODEL = "lgbm_solusdt_s_fw60_2021"
# MODEL = "lgbm_solusdt_s_fw60_2022"
# MODEL = "lgbm_solusdt_s_fw60_2023"
# MODEL = "lgbm_solusdt_s_fw60_2024"
# MODEL = "lgbm_solusdt_s_fw60_2025"

# --- Pipeline step kiválasztása ---
# Teljes pipeline (setup → feature_engineering → search → train):
cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL]

# Csak setup:
# cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL, "--step", "setup"]

# Csak feature engineering:
# cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL, "--step", "feature_engineering"]

# Csak search — stage opciók: smoke (gyors pilot) / explore / refine (hosszú)
# cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL, "--step", "search", "--stage", "smoke"]
# cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL, "--step", "search", "--stage", "explore"]
# cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL, "--step", "search", "--stage", "refine", "--n-trials", "200"]
# cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL, "--step", "search", "--stage", "refine", "--timeout-hours", "2"]

# Csak train:
# cmd = ["uv", "run", "python", "src/modeling/pipeline.py", "--model", MODEL, "--step", "train"]

subprocess.run(cmd, check=True)